
# Exercises XP : Evaluating LLMs for Summarization



## What you will learn
- Hands-on evaluation for summarization: accuracy vs. ROUGE.
- Strengths/weaknesses of metrics and model size comparisons.
- Using Hugging Face `transformers` + `evaluate` for quick experiments.
- Data loading, sampling, preprocessing, and debugging model outputs.

**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.


In [ ]:

# Part I. Setup (run once per runtime)
# Install minimal deps; keep quiet to reduce noise.
!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')



### Part II. Dataset loading and exploration
Preferred dataset: [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) (map `article` -> `prompt_text`, `highlights` -> `prompt_title`).
- If you have local train/test CSVs with `prompt_text` / `prompt_title`, set the paths below.
- Otherwise, we will auto-sample a small slice from the HF dataset to keep things light.
- Show a couple of rows for a sanity check.
If HF download fails, a tiny fallback sample is used.


In [ ]:

import pandas as pd
from datasets import load_dataset

# Point to your data; leave empty to use the HF cnn_dailymail sample or fallback
train_path = ''  # e.g., '/content/train.csv'
test_path = ''   # e.g., '/content/test.csv'

fallback = pd.DataFrame([
    {
        'prompt_text': 'The cat sat on the mat and purred loudly while the sun set.',
        'prompt_title': 'Cat rests on mat at sunset'
    },
    {
        'prompt_text': 'Scientists discovered water on the moon, opening new research paths.',
        'prompt_title': 'Water found on the moon'
    },
    {
        'prompt_text': 'The local team won the championship after a dramatic final match.',
        'prompt_title': 'Local team clinches title'
    },
])

def load_and_sample(path, split_name, n):
    if path:
        df = pd.read_csv(path)
    else:
        try:
            hf_split = f"{split_name}[:{max(n, 3)}]"
            ds = load_dataset('abisee/cnn_dailymail', '3.0.0', split=hf_split)
            df = ds.to_pandas()[['article', 'highlights']].rename(columns={'article': 'prompt_text', 'highlights': 'prompt_title'})
        except Exception as exc:
            print(f"HF load failed ({exc}); using tiny fallback sample.")
            df = fallback.copy()
    return df.sample(min(n, len(df)), random_state=42).reset_index(drop=True)

train_df = load_and_sample(train_path, 'train', 100)
test_df = load_and_sample(test_path, 'test', 50)

display(train_df.head(2))



### Part III. Summarization with T5 (implement)
Tasks:
- Write `batch_generator` to yield mini-batches.
- Write `summarize_with_t5` using `t5-small` (or swap sizes) with GPU if available.
- Prefix inputs with "summarize: " and decode with `skip_special_tokens=True`.
- Clear CUDA cache between batches (`torch.cuda.empty_cache()`) and gc.collect().


In [ ]:
import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration
from typing import Iterable, List

def batch_generator(items: List[str], batch_size: int):
    for i in range(0, len(items), batch_size):
        yield items[i : i + batch_size]

def summarize_with_t5(texts: List[str], model_name: str = 't5-small', batch_size: int = 4, max_new_tokens: int = 32):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

    all_summaries = []

    for batch_texts in batch_generator(texts, batch_size):
        # Prefix inputs with "summarize: "
        prefixed_texts = ["summarize: " + text for text in batch_texts]

        # Tokenize with prefix
        inputs = tokenizer(prefixed_texts, return_tensors='pt', padding=True, truncation=True)
        input_ids = inputs.input_ids.to(device)
        attention_mask = inputs.attention_mask.to(device)

        # Generate summaries
        outputs = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens
        )

        # Decode with skip_special_tokens=True
        decoded_summaries = [tokenizer.decode(g, skip_special_tokens=True) for g in outputs]
        all_summaries.extend(decoded_summaries)

        # Clear caches between batches
        del inputs, outputs, input_ids, attention_mask
        torch.cuda.empty_cache()
        gc.collect()

    return all_summaries

# RUN_FLAG keeps heavy generation optional for quick debugging
RUN_T5 = False
if RUN_T5:
    train_summaries_t5 = summarize_with_t5(train_df['prompt_text'].tolist(), model_name='t5-small', batch_size=2)
    display(pd.DataFrame({
        'prompt_text': train_df['prompt_text'],
        'reference_summary': train_df['prompt_title'],
        't5_small_summary': train_summaries_t5
    }).head())
else:
    print("Skipping T5 generation for speed. Set RUN_T5=True to execute.")


In [ ]:
print("Running T5-small summarization...")
train_summaries_t5_small = summarize_with_t5(
    train_df['prompt_text'].tolist(),
    model_name='t5-small',
    batch_size=2
)
train_df['t5_small_summary'] = train_summaries_t5_small
print("T5-small summarization complete.")


In [ ]:
print("Running T5-base summarization...")
train_summaries_t5_base = summarize_with_t5(
    train_df['prompt_text'].tolist(),
    model_name='t5-base',
    batch_size=2
)
train_df['t5_base_summary'] = train_summaries_t5_base
print("T5-base summarization complete.")


In [ ]:
print("Running GPT-2 summarization...")
train_summaries_gpt2 = summarize_with_gpt2(
    train_df['prompt_text'].tolist(),
    model_name='gpt2',
    batch_size=2
)
train_df['gpt2_summary'] = train_summaries_gpt2
print("GPT-2 summarization complete.")


### Computing ROUGE scores for all models

In [ ]:
print("Computing ROUGE scores for T5-small...")
train_df = compute_rouge_per_row(train_df, 't5_small_summary')
print("Computing ROUGE scores for T5-base...")
train_df = compute_rouge_per_row(train_df, 't5_base_summary')
print("Computing ROUGE scores for GPT-2...")
train_df = compute_rouge_per_row(train_df, 'gpt2_summary')

print("ROUGE score computation complete. Displaying combined ROUGE scores:")


In [ ]:
# Aggregate average ROUGE scores for comparison
rouge_comparison_dict = {
    't5-small': compute_rouge_score(train_df['t5_small_summary'].tolist(), train_df['prompt_title'].tolist()),
    't5-base': compute_rouge_score(train_df['t5_base_summary'].tolist(), train_df['prompt_title'].tolist()),
    'gpt2': compute_rouge_score(train_df['gpt2_summary'].tolist(), train_df['prompt_title'].tolist())
}

# Display the comparison table
display(compare_models(rouge_comparison_dict))


### Side-by-side comparison of generated summaries

In [ ]:
# Display side-by-side summaries
display(compare_models_summaries(
    train_df,
    ['t5_small_summary', 't5_base_summary', 'gpt2_summary']
).head())


Now, based on the ROUGE scores and the qualitative comparison of summaries, you can proceed to the 'Wrap-up' section to discuss your findings and answer the questions posed in cell `131468a9`.


### Part IV. Accuracy evaluation (toy, likely near zero)
Implement a naive accuracy that checks exact string match between generated and reference summaries.
Discuss why this is harsh for free-form text (almost always zero).


In [ ]:

from typing import List

def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)

if 'train_summaries_t5' in locals():
    acc = compute_accuracy(train_summaries_t5, train_df['prompt_title'].tolist())
    print(f"Exact-match accuracy: {acc:.4f}")
else:
    print("Accuracy skipped (no predictions).")



### Part V. ROUGE metric implementation
Use `evaluate.load("rouge")` and NLTK sentence tokenizer.
Preprocess by joining sentences with newlines for better ROUGE-L.


In [ ]:
import evaluate
from nltk.tokenize import sent_tokenize
from typing import List

rouge = evaluate.load('rouge')

def normalize_text(text):
    sents = sent_tokenize(text.strip())
    return "\n".join(sents)

def compute_rouge_score(preds: List[str], refs: List[str]):
    normalized_preds = [normalize_text(p) for p in preds]
    normalized_refs = [normalize_text(r) for r in refs]
    results = rouge.compute(predictions=normalized_preds, references=normalized_refs)
    return results

# Smoke test with identical strings and empty prediction
test_preds = ["alpha beta", "", "The cat sat."]
test_refs  = ["alpha beta", "reference text", "The cat sat."]
print("ROUGE sanity check (fill function first):")
print(compute_rouge_score(test_preds, test_refs))



### Part VI. Understanding ROUGE scores
Experiments to run (describe your findings in a text cell):
- Exact match vs. empty prediction.
- Effect of stemming: e.g., "running" vs. "run".
- N-gram overlap: see how ROUGE-1 vs. ROUGE-2 change with partial overlap.
- Symmetry: swap preds/refs and compare.



### Part VII. Comparing small and large models
Goals:
- Generate summaries with `t5-small`, `t5-base`, and `gpt2` (TL;DR style prompt).
- Compute ROUGE for each and store per-row scores.
- Implement `compute_rouge_per_row` to add ROUGE columns to a DataFrame.
- Implement `summarize_with_gpt2` with a TL;DR: prefix and max length guard.
Use small batches and low `max_new_tokens` to keep things snappy.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

def summarize_with_gpt2(texts: List[str], model_name: str = 'gpt2', batch_size: int = 2, max_new_tokens: int = 32):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    # GPT-2 does not have a special pad token by default, set it to eos_token
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

    all_summaries = []

    for batch_texts in batch_generator(texts, batch_size):
        # Prefix inputs with 'TL;DR:'
        prefixed_texts = [text + ' TL;DR:' for text in batch_texts]

        inputs = tokenizer(prefixed_texts, return_tensors='pt', padding=True, truncation=True).to(device)

        # Generate summaries
        outputs = model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False # For consistent output
        )

        # Decode generated tokens, skipping the input prompt and special tokens
        decoded_summaries = []
        for i, output in enumerate(outputs):
            # Decode from the end of the input_ids to get only the generated part
            generated_text = tokenizer.decode(output[len(inputs.input_ids[i]):], skip_special_tokens=True)
            decoded_summaries.append(generated_text.strip())

        all_summaries.extend(decoded_summaries)

        # Clear caches between batches
        del inputs, outputs
        torch.cuda.empty_cache()
        gc.collect()

    return all_summaries

def compute_rouge_per_row(df: pd.DataFrame, pred_col: str, ref_col: str = 'prompt_title'):
    rouge_scores = []
    for _, row in df.iterrows():
        pred = row[pred_col]
        ref = row[ref_col]
        # The `compute_rouge_score` expects lists, so wrap single strings
        scores = compute_rouge_score([str(pred)], [str(ref)]) # Ensure pred/ref are strings
        rouge_scores.append(scores)

    # Add ROUGE scores as new columns to the DataFrame
    df[f'{pred_col}_rouge1_fmeasure'] = [s['rouge1'] for s in rouge_scores]
    df[f'{pred_col}_rouge2_fmeasure'] = [s['rouge2'] for s in rouge_scores]
    df[f'{pred_col}_rougeL_fmeasure'] = [s['rougeL'] for s in rouge_scores]
    return df

RUN_COMPARE = False
if RUN_COMPARE and 'train_summaries_t5' in locals():
    # Add other model outputs then compute per-row ROUGE
    pass



### Part VIII. Comparing all models
Implement:
- `compare_models` to aggregate average ROUGE across models.
- `compare_models_summaries` to show side-by-side summaries.
Present the tables and discuss which model wins and why.


In [ ]:
import pandas as pd

def compare_models(rouge_dict):
    results = []
    for model_name, scores_dict in rouge_dict.items():
        row = {'model': model_name}
        for rouge_type, metrics in scores_dict.items():
            if isinstance(metrics, dict) and 'fmeasure' in metrics:
                row[f'{rouge_type}_fmeasure'] = metrics['fmeasure']
            else:
                # This handles the case where compute_rouge_score might return directly the fmeasure
                # if it was modified to do so, or if there's a misunderstanding of the exact output.
                # Assuming the original structure of rouge.compute output: {'rouge1': {'fmeasure': x}}
                row[f'{rouge_type}_fmeasure'] = metrics # Fallback if it's not a nested dict
        results.append(row)
    return pd.DataFrame(results).set_index('model')

def compare_models_summaries(df: pd.DataFrame, pred_cols: list):
    display_cols = ['prompt_text', 'prompt_title'] + pred_cols
    return df[display_cols]



## Wrap-up
- Which metrics felt most informative? Why?
- How did model size impact ROUGE and qualitative quality?
- Where did accuracy break down as a metric?
- How would you extend this to human eval or adversarial probes?
Write a short reflection here.
